# 04 — Optical–SAR Feature Fusion
**Owner: Person 4**

Colab training notebook for the audited Dual ResNet-18 + MLP fusion baseline. The encoders are frozen; only the fusion head is trained on BigEarthNet-MM. The required evaluation is optical-only vs SAR-only vs optical+SAR, reported honestly.

## 1. Colab setup

In [ ]:
!nvidia-smi
!pip install -q -r requirements.txt


In [ ]:
import os, sys, torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
sys.path.insert(0, os.getcwd())


## 2. Prepare the official BigEarthNet v2 index

The official v2 release stores S2 B02/B03/B04/B08 and S1 VV/VH as separate GeoTIFFs, with pairing/labels/splits in metadata.parquet. Run the helper once after the dataset is available.

In [ ]:
# Example — adjust these paths to your Colab/Drive dataset location.
!python scripts/prepare_bigearthnet_mm.py \
  --s2-root /content/BigEarthNet-S2 \
  --s1-root /content/BigEarthNet-S1 \
  --metadata /content/metadata.parquet \
  --out data/raw/bigearthnet_mm \
  --limit-per-split 2000


## 3. Smoke-check the dataloader

In [ ]:
from src.utils.io_utils import load_config
from src.preprocessing.dataset_loader import get_dataloader
config = load_config("configs/config.yaml")
loader = get_dataloader(task="fusion", split="train", dataset="bigearthnet_mm", config=config, batch_size=4)
batch = next(iter(loader))
print(batch["image_optical"].shape, batch["image_sar"].shape)
print(batch["label"][:2])


## 4. Train

In [ ]:
!python -m src.training.train_fusion --config configs/config.yaml --epochs 3

## 5. Three-way ablation

In [ ]:
!python scripts/run_fusion_ablation.py --config configs/config.yaml --split test

## 6. Verify checkpoint

In [ ]:
import os
ckpt = config["models"]["fusion"]["checkpoint"]
print(ckpt, os.path.getsize(ckpt) / 1024**2, "MB")
